# Fase 1: Extracción de Datos (Extract)
En esta fase del pipeline de datos, nos conectaremos y extraeremos información desde múltiples fuentes de origen heterogéneas:
- **Ventas**: Archivo CSV (`ventas.csv`)
- **Productos**: Archivo Excel (`productos.xlsx`)
- **Clientes**: Archivo JSON (`clientes.json`)
- **Inventario**: Base de datos SQLite (`inventario.db`)
- **Marketing**: Respuesta de API REST en formato JSON (`api_marketing_response.json`)

In [ ]:
import os
import json
import sqlite3
import pandas as pd


## 1. Configuración de Rutas de Datos
Definiremos las rutas relativas a nuestros archivos de datos. Dado que este notebook se encuentra en la carpeta `notebooks`, los archivos de datos se localizan en `../data/`.

In [ ]:
DATA_DIR = "../data"
VENTAS_CSV = os.path.join(DATA_DIR, "ventas.csv")
PRODUCTOS_XLSX = os.path.join(DATA_DIR, "productos.xlsx")
CLIENTES_JSON = os.path.join(DATA_DIR, "clientes.json")
INVENTARIO_DB = os.path.join(DATA_DIR, "inventario.db")
MARKETING_JSON = os.path.join(DATA_DIR, "api_marketing_response.json")


## 2. Extracción de Productos (Excel)
Leemos la hoja de productos desde el archivo Excel usando `pandas` y examinamos la información.

In [ ]:
print(f"Leyendo productos de: {PRODUCTOS_XLSX}")
df_productos = pd.read_excel(PRODUCTOS_XLSX)
print(f"Total registros extraídos: {len(df_productos)}")
df_productos.head()


## 3. Extracción de Ventas (CSV)
Extraemos la información de ventas desde el archivo delimitado por comas (CSV).

In [ ]:
print(f"Leyendo ventas de: {VENTAS_CSV}")
df_ventas = pd.read_csv(VENTAS_CSV)
print(f"Total registros extraídos: {len(df_ventas)}")
df_ventas.head()


## 4. Extracción de Clientes (JSON)
Leemos el listado de clientes en formato JSON utilizando Pandas.

In [ ]:
print(f"Leyendo clientes de: {CLIENTES_JSON}")
df_clientes = pd.read_json(CLIENTES_JSON)
print(f"Total registros extraídos: {len(df_clientes)}")
df_clientes.head()


## 5. Extracción de Inventario (SQLite)
Conectamos a la base de datos de inventario y extraemos las tablas de `inventario_actual` y `movimientos_inventario`.

In [ ]:
print(f"Conectando a base de datos de inventario: {INVENTARIO_DB}")
with sqlite3.connect(INVENTARIO_DB) as conn:
    df_inventario_actual = pd.read_sql_query("SELECT * FROM inventario_actual;", conn)
    df_movimientos = pd.read_sql_query("SELECT * FROM movimientos_inventario;", conn)

print(f"Existencias extraídas: {len(df_inventario_actual)}")
print(f"Movimientos extraídos: {len(df_movimientos)}")


In [ ]:
print("--- Primeras filas de Inventario Actual ---")
display(df_inventario_actual.head())
print("\n--- Primeras filas de Movimientos ---")
display(df_movimientos.head())


## 6. Extracción de Marketing (API JSON normalizado)
Leemos los datos de campañas de marketing simulados desde el archivo JSON de respuesta de la API REST, utilizando `pd.json_normalize` para aplanar la estructura.

In [ ]:
print(f"Leyendo datos de marketing de: {MARKETING_JSON}")
with open(MARKETING_JSON, encoding="utf-8") as f:
    payload = json.load(f)

df_marketing = pd.json_normalize(payload, record_path="campaigns")
print(f"Campañas de marketing extraídas: {len(df_marketing)}")
df_marketing.head()


## 7. Resumen de la Extracción
A continuación mostramos un resumen rápido de las fuentes extraídas y su estructura original.

In [ ]:
resumen = pd.DataFrame({
    "Dataset": ["Productos", "Ventas", "Clientes", "Inventario Actual", "Movimientos Inventario", "Marketing"],
    "Registros": [len(df_productos), len(df_ventas), len(df_clientes), len(df_inventario_actual), len(df_movimientos), len(df_marketing)],
    "Columnas": [len(df_productos.columns), len(df_ventas.columns), len(df_clientes.columns), len(df_inventario_actual.columns), len(df_movimientos.columns), len(df_marketing.columns)]
})
resumen
